<a href="https://colab.research.google.com/github/Sophie-X31/Contract-Due-Diligence/blob/main/Contract_Due_Diligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets
!pip install transformers
!pip install mineru

### Load & Clean Dataset

In [ ]:
!git clone https://github.com/TheAtticusProject/cuad.git

Cloning into 'cuad'...
remote: Enumerating objects: 30, done.
remote: Total 30 (delta 0), reused 0 (delta 0), pack-reused 30 (from 1)
Receiving objects: 100% (30/30), 17.78 MiB | 47.90 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [ ]:
!unzip cuad/data.zip -d cuad/

Archive:  cuad/data.zip
  inflating: cuad/CUADv1.json        
  inflating: cuad/test.json          
  inflating: cuad/train_separate_questions.json  


In [ ]:
import pandas as pd
import json

with open("cuad/CUADv1.json", "r") as f:
    cuad_data = json.load(f)

rows = []
for doc in cuad_data['data']:
    doc_title = doc.get('title', 'Unknown Document')

    for para in doc.get('paragraphs', []):
        qas = para.get('qas', [])

        for qa in qas:
            clause_label = qa.get('question', None)
            answers = qa.get('answers', [])

            if not answers:
                rows.append({
                    'document_title': doc_title,
                    'clause_text': "",
                    'clause_label': clause_label
                })

            for ans in answers:
                clause_text = ans.get('text', None)

                if clause_text:
                    rows.append({
                        'document_title': doc_title,
                        'clause_text': clause_text,
                        'clause_label': clause_label
                    })

df = pd.DataFrame(rows)

In [ ]:
categories = [
    "Document Name",
    "Parties",
    "Agreement Date",
    "Effective Date",
    "Expiration Date",
    "Renewal Term",
    "Notice Period To Terminate Renewal",
    "Governing Law",
    "Most Favored Nation",
    "Non-Compete",
    "Exclusivity",
    "No-Solicit of Customers",
    "Competitive Restriction Exception",
    "No-Solicit Of Employees",
    "Rofr/Rofo/Rofn",
    "Anti-Assignment",
    "Price Restrictions",
    "Minimum Commitment",
    "License Grant",
    "Post-Termination Services",
    "Warranty Duration",
    "Insurance",
    "Covenant Not to Sue",
    "Change of Control",
    "Audit Rights",
    "Uncapped Liability",
    "Cap On Liability",
    "Termination For Convenience",
    "Volume Restriction",
    "Ip Ownership Assignment",
    "Irrevocable Or Perpetual License",
    "Revenue/Profit Sharing",
    "Non-Transferable License",
    "Affiliate License-Licensee",
    "Liquidated Damages",
    "Joint Ip Ownership",
    "Third Party Beneficiary",

    "Unlimited/All-You-Can-Eat-License",
    "Source Code Escrow",
    "Affiliate License-Licensor",
    "Non-Disparagement"
]

def map_clause_label(label):
    for cat in categories:
        if cat.lower() in label.lower():
            return cat
    return label

df['clause_label'] = df['clause_label'].apply(map_clause_label)

In [ ]:
df.to_csv("/content/cuad_clauses_cleaned.csv", index=False)

### Train Classifier

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/Sophie-X31/Contract-Due-Diligence/main/cuad_clauses_cleaned.csv"
df = pd.read_csv(url)
df['clause_text'] = df['clause_text'].fillna("")
df = df[df['clause_text'].str.strip() != ""]

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
# Encode Labels
label_encoder = LabelEncoder()
df['label_id'] = label_encoder.fit_transform(df['clause_label'])
num_labels = len(label_encoder.classes_)

# Split Train/Test Data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['clause_text'].tolist(),
    df['label_id'].tolist(),
    test_size=0.1,
    random_state=42,
    stratify=df['label_id']
)

class ClauseDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in encoding.items()} | {'labels': torch.tensor(self.labels[idx])}

tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
train_dataset = ClauseDataset(train_texts, train_labels, tokenizer)
val_dataset = ClauseDataset(val_texts, val_labels, tokenizer)

In [ ]:
# Train Model
model = AutoModelForSequenceClassification.from_pretrained(
    "nlpaueb/legal-bert-base-uncased",
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir="./legalbert_clause_results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    logging_steps=50,
    save_steps=200,
    eval_strategy="steps",
    logging_dir="./logs",
    load_best_model_at_end=True,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

Step,Training Loss,Validation Loss
50,3.345167,3.045613
100,3.105168,2.754083
150,2.787950,2.472402
200,2.436976,2.170337
250,2.092404,1.795283
300,1.864566,1.552381
350,1.601033,1.458590
400,1.376958,1.297331
450,1.260941,1.190263
500,1.171749,1.111758


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3110, training_loss=0.8891498930584579, metrics={'train_runtime': 2839.9601, 'train_samples_per_second': 8.761, 'train_steps_per_second': 1.095, 'total_flos': 3274100715847680.0, 'train_loss': 0.8891498930584579, 'epoch': 2.0})

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

predictions = trainer.predict(val_dataset)
logits = predictions.predictions
labels = predictions.label_ids
preds = np.argmax(logits, axis=-1)

acc = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average='weighted')
print("Validation Accuracy:", acc)
print("Validation F1 (weighted):", f1)

Validation Accuracy: 0.8040491684743312
Validation F1 (weighted): 0.7805687681482475


In [ ]:
from sklearn.metrics import classification_report

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}

report = classification_report(labels, preds, target_names=[id2label[i] for i in range(len(id2label))])
print(report)

                                    precision    recall  f1-score   support

                    Agreement Date       0.62      0.98      0.76        48
                   Anti-Assignment       0.85      0.85      0.85        65
                      Audit Rights       0.98      0.95      0.97        64
                  Cap On Liability       0.72      0.97      0.83        67
                 Change of Control       0.74      0.68      0.71        25
               Covenant Not to Sue       0.79      0.88      0.83        17
                     Document Name       1.00      1.00      1.00        52
                    Effective Date       0.73      0.24      0.37        45
                       Exclusivity       0.00      0.00      0.00        12
                   Expiration Date       0.74      0.79      0.76        47
                     Governing Law       0.96      0.98      0.97        46
                         Insurance       0.95      0.95      0.95        56
           

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
output_dir = "./legalbert_clause_model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./legalbert_clause_model/tokenizer_config.json',
 './legalbert_clause_model/tokenizer.json')

In [ ]:
import joblib
joblib.dump(label_encoder, "label_encoder.pkl")

### Data Room

Contract Source: https://www.sec.gov/edgar/browse/?CIK=1730168&owner=exclude

Open Source Extractor: https://github.com/opendatalab/MinerU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import re

folder_path = "/content/drive/MyDrive/Side Project"

def extract_text_from_pdf_info(data):
    texts = []

    for page in data.get("pdf_info", []):
        for block in page.get("para_blocks", []):

            # normal text/title blocks
            if "lines" in block:
                for line in block["lines"]:
                    for span in line.get("spans", []):
                        if "content" in span:
                            texts.append(span["content"])

            # list blocks (nested)
            if block.get("type") == "list":
                for sub_block in block.get("blocks", []):
                    for line in sub_block.get("lines", []):
                        for span in line.get("spans", []):
                            if "content" in span:
                                texts.append(span["content"])
    return texts


def split_clauses(text):
    pattern = r'\n(?=(?:\d+\.\s|[A-Z]\.\s))'
    clauses = re.split(pattern, text)
    return [c.strip() for c in clauses if len(c.strip()) > 80]

def split_numbered_clauses(text):
    pattern = r'(?<=\n)(?=(?:\d+(?:\.\d+)*\.?\s+|[A-Z]\.\s+|\([a-z]\)\s+))'
    clauses = [c.strip() for c in re.split(pattern, text) if len(c.strip()) > 50]
    return clauses

In [ ]:
all_clauses = {}
for file in os.listdir(folder_path):
    if file.lower().endswith(".json"):
        path = os.path.join(folder_path, file)

        with open(path, "r") as f:
            data = json.load(f)

        texts = extract_text_from_pdf_info(data)
        full_text = "\n".join(texts)
        clauses = split_clauses(full_text)
        clauses = split_numbered_clauses(full_text)
        all_clauses[file] = clauses

In [ ]:
print("Document Titles:")
for doc in all_clauses.keys():
  print(doc, " : number of clauses = ", len(all_clauses[doc]))

# first = list(all_clauses.keys())[1]
# check = all_clauses[first][:10]
# print(*check, sep='\n\n')

Document Titles:
First Amendment to Lease Agreement (EX-10.18).json  : number of clauses =  12
Lease Agreement FIVE POINT OFFICE VENTURE I (EX-10.29).json  : number of clauses =  219
Performance Stock Unit Award Agreement (EX-10.1).json  : number of clauses =  53
Restricted Stock Unit Award Agreement (EX-10.1).json  : number of clauses =  128
Serverance Benefit Agreement (EX-10.1).json  : number of clauses =  38


### LLM Risk Analysis

#### Label Clauses

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm import tqdm
import joblib

label_encoder = joblib.load("label_encoder.pkl")

model_path = "/content/drive/MyDrive/Side Project/legalbert_clause_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

def classify_clauses(clauses, batch_size=8, threshold=0.7):
    model.eval()

    results = []
    for i in tqdm(range(0, len(clauses), batch_size), desc="Classifying clauses"):
        batch = clauses[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)

        for clause, pred, prob in zip(batch, preds, probs):
            confidence = prob[pred].item()
            label = label_encoder.inverse_transform([pred.item()])[0] if confidence >= threshold else None
            results.append({
                "clause": clause,
                "label": label,
                "confidence": confidence
            })

    return results

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
import torch
classified_results = {}
for file, clauses in all_clauses.items():
    classified_results[file] = classify_clauses(clauses)

Classifying clauses: 100%|██████████| 5/5 [00:23<00:00,  4.73s/it]


In [ ]:
# print("Document Titles:")
# for doc in classified_results.keys():
#   print(doc, " : number of clauses = ", len(classified_results[doc]))

first = list(classified_results.keys())[0]
check = classified_results[first][:7]
print(*check, sep='\n\n')

{'clause': 'FIRST AMENDMENT TO LEASE AGREEMENT\nTHIS FIRST AMENDMENT TO LEASE AGREEMENT ("Amendment") is dated for reference purposes as of May 22, 2020, but is deemed entered into and effective as of the "Effective Date" (defined below), and is made by and between FIVE. POINT OFFICE VENTURE I, LLC, a Delaware limited liability company ("Landlord"), and BROADCOM CORPORATION, a California corporation ("Tenant").\nRECITALS:', 'label': 'Effective Date', 'confidence': 0.7384604811668396}

{'clause': 'A. Landlord and Tenant are parties to that certain Lease Agreement dated August 10, 2017 (the "Lease"), pursuant to which Tenant leases from Landlord certain premises located in Irvine, California, as more particularly described in the Lease (the "Premises").', 'label': None, 'confidence': 0.4449581801891327}

{'clause': 'B. Landlord is selling a portion of the Project to City of Hope ("COH") for the development and operation of a cancer center within an existing Building (referred to as Build

In [ ]:
import json
with open("/content/drive/MyDrive/Side Project/classified_results.json", "w") as f:
    json.dump(classified_results, f, indent=2)

#### LLM Reasoning

In [ ]:
import json
with open("/content/drive/MyDrive/Side Project/classified_results.json", "r") as f:
    classified_results = json.load(f)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd

model_name = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [ ]:
import re

def parse_risk_response(response):
    response = response.strip()

    # Format 1: Risk level + Explanation
    match = re.search(
        r'Risk\s*level\s*[:\-]?\s*(\w+).*?Explanation\s*[:\-]?\s*(.+)',
        response,
        flags=re.I | re.S
    )
    if match:
        risk_level = match.group(1).capitalize()
        explanation = match.group(2).strip()
        return risk_level, explanation

    # Format 2: Level: Explanation
    match = re.match(r'(\w+)\s*[:\-]\s*(.+)', response)
    if match:
        risk_level = match.group(1).capitalize()
        explanation = match.group(2).strip()
        return risk_level, explanation

    # Format 3: Risk Level: Low. Explanation in same sentence
    match = re.search(r'Risk\s*Level\s*[:\-]?\s*(\w+)\.\s*(.+)', response, flags=re.I | re.S)
    if match:
        risk_level = match.group(1).capitalize()
        explanation = match.group(2).strip()
        return risk_level, explanation

    # Fallback
    return "Response Format Error", response

In [ ]:
import torch
from transformers import pipeline
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "temperature": 0.7,
    "do_sample": True,
}

def assess_risk_with_phi3(clauses_data, batch_size=8):
    risky_clauses = []
    for i in tqdm(range(0, len(clauses_data), batch_size), desc="Assessing risk"):
        batch = clauses_data[i:i+batch_size]

        for item in batch:
            # Prepare prompt for each clause
            clause_text = item["clause"]
            label = item["label"]
            confidence = item["confidence"]
            prompt = (
              f"Clause: {clause_text}\n"
              f"Classifier label: {label}, confidence: {confidence:.2f}\n\n"
              f"Analyze the risk associated with this clause and categorize it as No, Low, Medium, or High risk.\n"
              f"Provide a one-line explanation for any risk level above No risk.\n"
              f"Output format: 'Risk Level: explanation.'"
            )
            messages = [
                {"role": "system", "content": "You are a legal and investment expert performing contract due diligence."},
                {"role": "user", "content": prompt},
            ]

            # Send prompt to PHI-3 mini
            output = pipe(messages, **generation_args)
            response = output[0]['generated_text']

            # Parse response
            risk_level, explanation = parse_risk_response(response)
            risky_clauses.append({
                "clause": item["clause"],
                "risk_level": risk_level,
                "explanation": explanation,
                "label": label,
                "confidence": confidence
            })

    df = pd.DataFrame(risky_clauses)
    counts = df["risk_level"].value_counts().to_dict()
    counts = {k: counts.get(k, 0) for k in ["Low","Medium","High"]}
    return counts, df

In [ ]:
# document_risk_analysis = {}
# for doc_name, clauses_list in classified_results.items():
#     counts, df_risky = assess_risk_with_phi3(clauses_list)
#     document_risk_analysis[doc_name] = (counts, df_risky)

first_doc_name = list(classified_results.keys())[0]
first_clauses_list = classified_results[first_doc_name]
counts, df_risky = assess_risk_with_phi3(first_clauses_list)
document_risk_analysis = {first_doc_name: (counts, df_risky)}

Assessing risk:   0%|          | 0/2 [00:00<?, ?it/s]Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/tr

In [ ]:
print("Document:", first_doc_name)
print("Risk counts:", counts)
# print(df_risky.iloc[1]["risk_level"])
# print(df_risky.iloc[1]["explanation"])
df_risky.head(12)

# for doc, (counts, df) in document_risk_analysis.items():
#     print(doc, counts)
#     print(df.head())

Document: First Amendment to Lease Agreement (EX-10.18).json
Risk counts: {'Low': 7, 'Medium': 5, 'High': 0}


,clause,risk_level,explanation,label,confidence
0,FIRST AMENDMENT TO LEASE AGREEMENT\nTHIS FIRST...,Medium,There is medium risk because the amendment's r...,Effective Date,0.738460
1,A. Landlord and Tenant are parties to that cer...,Low,The described clause simply confirms the exist...,None,0.444958
2,B. Landlord is selling a portion of the Projec...,Low,The risk associated with this clause appears l...,None,0.311725
3,C. Landlord and Tenant desire to amend the Lea...,Low,The clause allows for amendments but does not ...,None,0.194067
4,1. Effective Date/Incorporation. This Amendmen...,Low,The risk is considered low because the effecti...,Effective Date,0.837517
5,2. Campus REA. From and after the Effective Da...,Low,The risk associated with requiring a Tenant Re...,None,0.278177
6,3. Development CC&Rs. From and after the Effec...,Low,This clause limits the Landlord's ability to u...,None,0.263401
7,"4. Tenant's Share. As of the Effective Date, S...",Medium,The clause introduces variability in Tenant's ...,Revenue/Profit Sharing,0.907833
8,5. Landlord's Consent for Transfer Notices. As...,Medium,The clause imposes a strict timeline for Landl...,None,0.592386
9,(a) Effect of Amendment. Except to the extent ...,Low,The clause implies that any amendments to the ...,None,0.311727


In [ ]:
import json
file_path = "/content/drive/MyDrive/Side Project/document_risk_analysis.json"
with open(file_path, "w") as f:
    json.dump(document_risk_analysis, f, indent=2)

#### View Results

In [ ]:
file_path = "/content/drive/MyDrive/Side Project/document_risk_analysis.json"
with open(file_path, "r") as f:
    document_risk_analysis = json.load(f)